In [1]:
!nvidia-smi --query-gpu=name,uuid,compute_cap,memory.total --format=csv
!mpirun --version | head -n 2
!pip install -q mpi4py numba numpy

name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-beadeb20-0fa1-8ad8-bf3f-4437b48312ea, 7.5, 15360 MiB
mpirun (Open MPI) 4.1.6



In [4]:
# STEP 2: STUDENT IDENTITY CONFIGURATION

GITHUB_USERNAME = "belkamenbirgee"

print("GitHub username:", GITHUB_USERNAME)

assert len(GITHUB_USERNAME.strip()) > 0

print("Student identity configured successfully!")

Student identity configured successfully!


In [10]:

%%writefile distributed_cuda_solver.py

import os
import sys
import json
import math
import hashlib
import subprocess
import numpy as np

from mpi4py import MPI
from numba import cuda, float64

# ============================================================
# GLOBAL CONSTANTS
# ============================================================

TILE_DIM = 16
SHARED_DIM = TILE_DIM + 2

GLOBAL_NX = 512
GLOBAL_NY = 512

LOCAL_NX = 256
LOCAL_NY = 256

STEPS = 400
LAMBDA_DAMP = 0.15

DX = 1.0 / 513
DY = 1.0 / 513

# ============================================================
# GPU UUID
# ============================================================

def get_gpu_uuid():
    out = subprocess.check_output(
        [
            "nvidia-smi",
            "--query-gpu=uuid",
            "--format=csv,noheader"
        ],
        encoding="utf-8"
    )
    return out.strip().splitlines()[0].strip()


# ============================================================
# STUDENT SEED
# ============================================================

def derive_student_seed(github_user, gpu_uuid):
    digest = hashlib.sha256(
        f"{github_user.strip()}::{gpu_uuid}".encode("utf-8")
    ).hexdigest()

    return int(digest[:16], 16)


# ============================================================
# TASK 1: MPI DISTRIBUTED TOPOLOGY
# ============================================================

def init_distributed_context(github_user):

    comm = MPI.COMM_WORLD

    rank = comm.Get_rank()
    size = comm.Get_size()

    if size != 4:
        raise RuntimeError(
            f"Expected exactly 4 MPI ranks, got {size}"
        )

    # Create a 2 x 2 Cartesian process grid
    dims = [2, 2]
    periods = [False, False]

    cart_comm = comm.Create_cart(
        dims=dims,
        periods=periods,
        reorder=False
    )

    # Get Cartesian coordinates
    coords = cart_comm.Get_coords(rank)
    row_rank, col_rank = coords

    # Find neighboring ranks
    nbr_up, nbr_down = cart_comm.Shift(0, 1)
    nbr_left, nbr_right = cart_comm.Shift(1, 1)

    # Select GPU 0
    cuda.select_device(0)

    # Get GPU UUID on rank 0 and share it
    gpu_uuid = get_gpu_uuid() if rank == 0 else None
    gpu_uuid = cart_comm.bcast(gpu_uuid, root=0)

    # Generate hardware-bound seed
    instance_seed = derive_student_seed(
        github_user,
        gpu_uuid
    )

    print(
        f"Rank {rank}: coordinates={coords}, "
        f"neighbors={(nbr_up, nbr_down, nbr_left, nbr_right)}"
    )

    return (
        cart_comm,
        rank,
        size,
        row_rank,
        col_rank,
        nbr_up,
        nbr_down,
        nbr_left,
        nbr_right,
        instance_seed
    )


Writing distributed_cuda_solver.py


In [11]:
import os

filename = "distributed_cuda_solver.py"

print("File exists:", os.path.exists(filename))

if os.path.exists(filename):
    print("File size:", os.path.getsize(filename), "bytes")

File exists: True
File size: 2653 bytes


In [12]:

%%writefile -a distributed_cuda_solver.py

# ============================================================
# TASK 2: SEEDED HETEROGENEOUS SUBDOMAIN INITIALIZATION
# ============================================================

def initialize_subdomains(
    row_rank,
    col_rank,
    instance_seed,
    nbr_up,
    nbr_down,
    nbr_left,
    nbr_right
):

    # Each MPI rank owns 256 x 256 cells
    ny = LOCAL_NY
    nx = LOCAL_NX

    # Local padded coordinates: 0..257
    local_y = np.arange(ny + 2)
    local_x = np.arange(nx + 2)

    # Convert local indices to global physical coordinates
    global_y = row_rank * ny + local_y - 1
    global_x = col_rank * nx + local_x - 1

    y = (global_y + 1) * DY
    x = (global_x + 1) * DX

    X, Y = np.meshgrid(x, y)

    # Rank-specific deterministic random generator
    rng = np.random.default_rng(
        (int(instance_seed) + row_rank * 2 + col_rank) % (2**64)
    )

    # --------------------------------------------------------
    # Spatially heterogeneous diffusivity: 0.08 to 0.24
    # --------------------------------------------------------

    kappa = (
        0.08
        + 0.16 * (
            0.5
            + 0.5 * np.sin(2 * np.pi * X)
            * np.cos(2 * np.pi * Y)
        )
    )

    kappa = np.clip(kappa, 0.08, 0.24)
    kappa = kappa.astype(np.float64)

    # --------------------------------------------------------
    # Initial temperature: seeded Gaussian thermal pulses
    # --------------------------------------------------------

    u = np.zeros((ny + 2, nx + 2), dtype=np.float64)

    number_of_pulses = 3

    for pulse in range(number_of_pulses):

        center_x = rng.uniform(0.2, 0.8)
        center_y = rng.uniform(0.2, 0.8)

        amplitude = rng.uniform(0.5, 1.0)

        sigma = rng.uniform(0.04, 0.10)

        gaussian = amplitude * np.exp(
            -(
                (X - center_x) ** 2
                + (Y - center_y) ** 2
            ) / (2 * sigma ** 2)
        )

        u += gaussian

    # --------------------------------------------------------
    # Localized source field
    # --------------------------------------------------------

    source_x = 0.5
    source_y = 0.5
    source_sigma = 0.08

    source = np.exp(
        -(
            (X - source_x) ** 2
            + (Y - source_y) ** 2
        ) / (2 * source_sigma ** 2)
    )

    source = source.astype(np.float64)

    # --------------------------------------------------------
    # Enforce physical outer boundary conditions
    # --------------------------------------------------------

    if nbr_up == MPI.PROC_NULL:
        u[0, :] = 0.0

    if nbr_down == MPI.PROC_NULL:
        u[ny + 1, :] = 0.0

    if nbr_left == MPI.PROC_NULL:
        u[:, 0] = 0.0

    if nbr_right == MPI.PROC_NULL:
        u[:, nx + 1] = 0.0

    return u, kappa, source



Appending to distributed_cuda_solver.py


In [13]:

%%writefile -a distributed_cuda_solver.py

# ============================================================
# TASK 3: GPU HALO PACK / UNPACK KERNELS
# ============================================================

@cuda.jit
def pack_halos_kernel(u, send_buffer, direction, nx, ny):

    idx = cuda.grid(1)

    if idx < max(nx, ny):

        if direction == 0 and idx < nx:
            send_buffer[idx] = u[1, idx + 1]

        elif direction == 1 and idx < nx:
            send_buffer[idx] = u[ny, idx + 1]

        elif direction == 2 and idx < ny:
            send_buffer[idx] = u[idx + 1, 1]

        elif direction == 3 and idx < ny:
            send_buffer[idx] = u[idx + 1, nx]


@cuda.jit
def unpack_halos_kernel(u, recv_buffer, direction, nx, ny):

    idx = cuda.grid(1)

    if idx < max(nx, ny):

        if direction == 0 and idx < nx:
            u[0, idx + 1] = recv_buffer[idx]

        elif direction == 1 and idx < nx:
            u[ny + 1, idx + 1] = recv_buffer[idx]

        elif direction == 2 and idx < ny:
            u[idx + 1, 0] = recv_buffer[idx]

        elif direction == 3 and idx < ny:
            u[idx + 1, nx + 1] = recv_buffer[idx]


# ============================================================
# TASK 3: ASYNCHRONOUS MPI HALO EXCHANGE
# ============================================================

def exchange_halos_async(
    u,
    cart_comm,
    nbr_up,
    nbr_down,
    nbr_left,
    nbr_right,
    stream_boundary
):

    nx = LOCAL_NX
    ny = LOCAL_NY

    neighbors = [
        nbr_up,
        nbr_down,
        nbr_left,
        nbr_right
    ]

    # Directions: 0=up, 1=down, 2=left, 3=right
    lengths = [nx, nx, ny, ny]

    # Opposite-direction message tags
    send_tags = [0, 1, 2, 3]
    recv_tags = [1, 0, 3, 2]

    max_length = max(nx, ny)

    # Allocate pinned host buffers
    host_send = [
        cuda.pinned_array(max_length, dtype=np.float64)
        for _ in range(4)
    ]

    host_recv = [
        cuda.pinned_array(max_length, dtype=np.float64)
        for _ in range(4)
    ]

    # Allocate GPU buffers
    device_send = [
        cuda.device_array(max_length, dtype=np.float64)
        for _ in range(4)
    ]

    device_recv = [
        cuda.device_array(max_length, dtype=np.float64)
        for _ in range(4)
    ]

    # --------------------------------------------------------
    # Stage A: GPU packing and asynchronous D2H transfers
    # --------------------------------------------------------

    threads = 256
    blocks = (max_length + threads - 1) // threads

    for direction in range(4):

        pack_halos_kernel[blocks, threads, stream_boundary](
            u,
            device_send[direction],
            direction,
            nx,
            ny
        )

        device_send[direction].copy_to_host(
            host_send[direction],
            stream=stream_boundary
        )

    stream_boundary.synchronize()

    # Initialize receive buffers to zero.
    # This preserves zero values for MPI.PROC_NULL neighbors.
    for direction in range(4):
        host_recv[direction][:] = 0.0

    # --------------------------------------------------------
    # Stage B: Non-blocking MPI communication
    # --------------------------------------------------------

    requests = []

    # Post receives first
    for direction in range(4):

        request = cart_comm.Irecv(
            host_recv[direction][:lengths[direction]],
            source=neighbors[direction],
            tag=recv_tags[direction]
        )

        requests.append(request)

    # Post sends
    for direction in range(4):

        request = cart_comm.Isend(
            host_send[direction][:lengths[direction]],
            dest=neighbors[direction],
            tag=send_tags[direction]
        )

        requests.append(request)

    MPI.Request.Waitall(requests)

    # --------------------------------------------------------
    # Stage C: Asynchronous H2D transfers and unpacking
    # --------------------------------------------------------

    for direction in range(4):

        device_recv[direction].copy_to_device(
            host_recv[direction],
            stream=stream_boundary
        )

        unpack_halos_kernel[blocks, threads, stream_boundary](
            u,
            device_recv[direction],
            direction,
            nx,
            ny
        )

    stream_boundary.synchronize()

    return host_send, host_recv


Appending to distributed_cuda_solver.py


In [14]:

%%writefile -a distributed_cuda_solver.py

# ============================================================
# TASK 4: 2D SHARED-MEMORY TILED CUDA STENCIL
# ============================================================

@cuda.jit
def tiled_stencil_kernel(
    u,
    u_next,
    kappa,
    source,
    dt,
    dx,
    dy,
    r_start,
    r_end,
    c_start,
    c_end
):

    # Shared-memory tile: 16x16 cells + 1-cell apron
    smem = cuda.shared.array(
        shape=(18, 18),
        dtype=float64
    )

    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y

    row = cuda.blockIdx.y * 16 + ty + 1
    col = cuda.blockIdx.x * 16 + tx + 1

    # Load center cells
    smem[ty + 1, tx + 1] = u[row, col]

    # Load top and bottom apron
    if ty == 0:
        smem[0, tx + 1] = u[row - 1, col]

    if ty == 15:
        smem[17, tx + 1] = u[row + 1, col]

    # Load left and right apron
    if tx == 0:
        smem[ty + 1, 0] = u[row, col - 1]

    if tx == 15:
        smem[ty + 1, 17] = u[row, col + 1]

    # All threads must finish loading before computation
    cuda.syncthreads()

    # Check requested computation region
    if (
        row >= r_start and row <= r_end
        and col >= c_start and col <= c_end
    ):

        center = smem[ty + 1, tx + 1]

        north = smem[ty, tx + 1]
        south = smem[ty + 2, tx + 1]
        west = smem[ty + 1, tx]
        east = smem[ty + 1, tx + 2]

        # Arithmetic-average interface conductivities
        k_north = 0.5 * (kappa[row - 1, col] + kappa[row, col])
        k_south = 0.5 * (kappa[row + 1, col] + kappa[row, col])

        k_west = 0.5 * (kappa[row, col - 1] + kappa[row, col])
        k_east = 0.5 * (kappa[row, col + 1] + kappa[row, col])

        # Anisotropic diffusion terms
        diffusion_x = (
            k_south * (south - center)
            - k_north * (center - north)
        ) / (dx * dx)

        diffusion_y = (
            k_east * (east - center)
            - k_west * (center - west)
        ) / (dy * dy)

        # Reaction-diffusion update
        reaction = -0.15 * center * center * center

        result = center + dt * (
            diffusion_x
            + diffusion_y
            + reaction
            + source[row, col]
        )

        u_next[row, col] = result


# ============================================================
# TASK 4: KERNEL LAUNCH CONFIGURATION
# ============================================================

THREADS_2D = (16, 16)
BLOCKS_2D = (16, 16)


def launch_stencil_region(
    u,
    u_next,
    kappa,
    source,
    dt,
    stream,
    r_start,
    r_end,
    c_start,
    c_end
):

    tiled_stencil_kernel[
        BLOCKS_2D,
        THREADS_2D,
        stream
    ](
        u,
        u_next,
        kappa,
        source,
        dt,
        DX,
        DY,
        r_start,
        r_end,
        c_start,
        c_end
    )


Appending to distributed_cuda_solver.py


In [15]:
%%writefile -a distributed_cuda_solver.py

# ==============================================================================
# TASK 4: SHARED-MEMORY TILED CUDA STENCIL
# ==============================================================================

@cuda.jit
def tiled_stencil_kernel(u, kappa, source, out, dt,
                         r_start, r_end, c_start, c_end):
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y

    row = cuda.blockIdx.y * 16 + ty + 1
    col = cuda.blockIdx.x * 16 + tx + 1

    smem = cuda.shared.array((18, 18), dtype=float64)

    # Load center cells
    smem[ty + 1, tx + 1] = u[row, col]

    # Load halo apron
    if tx == 0:
        smem[ty + 1, 0] = u[row, col - 1]

    if tx == 15:
        smem[ty + 1, 17] = u[row, col + 1]

    if ty == 0:
        smem[0, tx + 1] = u[row - 1, col]

    if ty == 15:
        smem[17, tx + 1] = u[row + 1, col]

    # Load tile corners
    if tx == 0 and ty == 0:
        smem[0, 0] = u[row - 1, col - 1]

    if tx == 15 and ty == 0:
        smem[0, 17] = u[row - 1, col + 1]

    if tx == 0 and ty == 15:
        smem[17, 0] = u[row + 1, col - 1]

    if tx == 15 and ty == 15:
        smem[17, 17] = u[row + 1, col + 1]

    cuda.syncthreads()

    if (row >= r_start and row <= r_end and
        col >= c_start and col <= c_end):

        center = smem[ty + 1, tx + 1]
        north = smem[ty, tx + 1]
        south = smem[ty + 2, tx + 1]
        west = smem[ty + 1, tx]
        east = smem[ty + 1, tx + 2]

        kc = kappa[row, col]

        kn = 0.5 * (kc + kappa[row - 1, col])
        ks = 0.5 * (kc + kappa[row + 1, col])
        kw = 0.5 * (kc + kappa[row, col - 1])
        ke = 0.5 * (kc + kappa[row, col + 1])

        dx = 1.0 / 513.0
        dy = 1.0 / 513.0

        diffusion_x = (
            ke * (east - center) -
            kw * (center - west)
        ) / (dx * dx)

        diffusion_y = (
            ks * (south - center) -
            kn * (center - north)
        ) / (dy * dy)

        damping = 0.15 * center * center * center

        out[row, col] = center + dt * (
            diffusion_x + diffusion_y - damping + source[row, col]
        )


# ==============================================================================
# TASK 4: TIME-STEP ORCHESTRATOR
# ==============================================================================

def step_overlapped(u, kappa, source, cart_comm,
                    nbr_up, nbr_down, nbr_left, nbr_right,
                    stream_boundary, stream_interior, dt):

    # Exchange halo cells
    exchange_halos_async(
        u, cart_comm,
        nbr_up, nbr_down, nbr_left, nbr_right,
        stream_boundary
    )

    out = cuda.device_array_like(u)

    threads = (16, 16)
    blocks = (16, 16)

    # Deep interior
    tiled_stencil_kernel[blocks, threads, stream_interior](
        u, kappa, source, out, dt,
        2, 255, 2, 255
    )

    # Boundary ring
    tiled_stencil_kernel[blocks, threads, stream_boundary](
        u, kappa, source, out, dt,
        1, 256, 1, 256
    )

    stream_interior.synchronize()
    stream_boundary.synchronize()

    return out


# ==============================================================================
# TASK 5: ENERGY, RESIDUAL, PTX AUDIT AND SOLVER
# ==============================================================================

def calculate_metrics(u, previous_u):
    host_u = u.copy_to_host()
    host_previous = previous_u.copy_to_host()

    local_energy = float(
        np.sum(host_u[1:257, 1:257]) * DX * DY
    )

    local_residual = float(
        np.sum(
            (host_u[1:257, 1:257] -
             host_previous[1:257, 1:257]) ** 2
        )
    )

    return local_energy, local_residual


def run_solver(github_user):
    (
        cart_comm, rank, size,
        row_rank, col_rank,
        nbr_up, nbr_down, nbr_left, nbr_right,
        instance_seed
    ) = init_distributed_context(github_user)

    u_host, kappa_host, source_host = initialize_subdomains(
        row_rank, col_rank, instance_seed,
        nbr_up, nbr_down, nbr_left, nbr_right
    )

    u_initial = cuda.to_device(u_host)
    kappa = cuda.to_device(kappa_host)
    source = cuda.to_device(source_host)

    stream_boundary = cuda.stream()
    stream_interior = cuda.stream()

    dx = 1.0 / 513.0
    kappa_max = float(np.max(kappa_host))
    dt_max = dx * dx / (4.0 * kappa_max)
    dt = 0.2 * dt_max

    print(
        f"Rank {rank}: seed={instance_seed}, "
        f"kappa_max={kappa_max:.8f}, dt={dt:.12e}"
    )

    # Run the serialized and overlapped modes from identical initial states
    results = {}

    for mode in ["serialized", "overlapped"]:
        u = cuda.to_device(u_host)
        previous_u = cuda.to_device(u_host)

        start_time = MPI.Wtime()

        for step in range(1, STEPS + 1):
            previous_u = cuda.to_device(u.copy_to_host())

            if mode == "serialized":
                exchange_halos_async(
                    u, cart_comm,
                    nbr_up, nbr_down, nbr_left, nbr_right,
                    stream_boundary
                )

                out = cuda.device_array_like(u)

                tiled_stencil_kernel[(16, 16), (16, 16), stream_boundary](
                    u, kappa, source, out, dt,
                    1, 256, 1, 256
                )

                stream_boundary.synchronize()
                u = out

            else:
                u = step_overlapped(
                    u, kappa, source, cart_comm,
                    nbr_up, nbr_down, nbr_left, nbr_right,
                    stream_boundary, stream_interior, dt
                )

            if step % 100 == 0:
                local_energy, local_residual = calculate_metrics(
                    u, previous_u
                )

                global_energy = cart_comm.allreduce(
                    local_energy, op=MPI.SUM
                )

                global_residual = cart_comm.allreduce(
                    local_residual, op=MPI.SUM
                )

                if rank == 0:
                    print(
                        f"{mode} | Step {step:3d} | "
                        f"Energy={global_energy:.12e} | "
                        f"L2 residual={math.sqrt(global_residual):.12e}"
                    )

        elapsed = MPI.Wtime() - start_time
        results[mode] = {
            "u": u,
            "elapsed": elapsed
        }

        print(
            f"Rank {rank}: {mode} completed in "
            f"{elapsed * 1000:.3f} ms"
        )

    # Compare final local states
    serialized_host = results["serialized"]["u"].copy_to_host()
    overlapped_host = results["overlapped"]["u"].copy_to_host()

    local_difference = float(
        np.max(np.abs(serialized_host - overlapped_host))
    )

    max_difference = cart_comm.allreduce(
        local_difference, op=MPI.MAX
    )

    serialized_ms = cart_comm.allreduce(
        results["serialized"]["elapsed"], op=MPI.MAX
    ) * 1000.0

    overlapped_ms = cart_comm.allreduce(
        results["overlapped"]["elapsed"], op=MPI.MAX
    ) * 1000.0

    # PTX inspection
    ptx_counts = {
        "st.shared.f64": 0,
        "ld.shared.f64": 0,
        "bar.sync": 0
    }

    try:
        sample = cuda.to_device(np.zeros((258, 258), dtype=np.float64))
        output = cuda.device_array_like(sample)

        tiled_stencil_kernel[(16, 16), (16, 16)](
            sample, sample, sample, output, dt,
            1, 256, 1, 256
        )
        cuda.synchronize()

        signatures = list(tiled_stencil_kernel.overloads.keys())

        if signatures:
            ptx = tiled_stencil_kernel.inspect_asm(signatures[0])

            ptx_counts["st.shared.f64"] = ptx.count("st.shared.f64")
            ptx_counts["ld.shared.f64"] = ptx.count("ld.shared.f64")
            ptx_counts["bar.sync"] = ptx.count("bar.sync")

    except Exception as error:
        print("PTX audit error:", error)

    if rank == 0:
        manifest = {
            "github_username": github_user,
            "instance_seed": int(instance_seed),
            "mpi_ranks": size,
            "process_grid": [2, 2],
            "global_grid": [512, 512],
            "local_grid": [256, 256],
            "steps": STEPS,
            "dt": float(dt),
            "dt_max": float(dt_max),
            "kappa_max": float(kappa_max),
            "serialized_ms": float(serialized_ms),
            "overlapped_ms": float(overlapped_ms),
            "parity_max_diff": float(max_difference),
            "ptx_counts": ptx_counts,
            "ptx_audit_passed": all(
                value > 0 for value in ptx_counts.values()
            )
        }

        with open("submission_manifest.json", "w") as file:
            json.dump(manifest, file, indent=4)

        print("\nFINAL RESULTS")
        print(json.dumps(manifest, indent=4))

    cart_comm.Barrier()


# ==============================================================================
# PROGRAM ENTRY POINT
# ==============================================================================

if __name__ == "__main__":
    if len(sys.argv) < 2:
        raise RuntimeError(
            "Usage: python3 distributed_cuda_solver.py GITHUB_USERNAME"
        )

    run_solver(sys.argv[1])

Appending to distributed_cuda_solver.py


In [17]:
import py_compile

try:
    py_compile.compile("distributed_cuda_solver.py", doraise=True)
    print("SUCCESS: No Python syntax errors.")
except Exception as e:
    print("ERROR:", e)

SUCCESS: No Python syntax errors.


In [18]:
import numpy as np
from mpi4py import MPI
from numba import cuda

print("NumPy:", np.__version__)
print("MPI version:", MPI.Get_version())
print("MPI processes:", MPI.COMM_WORLD.Get_size())

print("\nCUDA available:", cuda.is_available())

if cuda.is_available():
    print("GPU:", cuda.get_current_device().name)

NumPy: 2.4.6
MPI version: (3, 1)
MPI processes: 1

CUDA available: True
GPU: Tesla T4


In [19]:
!which mpiexec
!mpiexec --version

/usr/bin/mpiexec
mpiexec (OpenRTE) 4.1.6

Report bugs to http://www.open-mpi.org/community/help/


In [23]:
%%writefile mpi_test.py
from mpi4py import MPI

comm = MPI.COMM_WORLD

print(
    f"Hello from rank {comm.Get_rank()} "
    f"of {comm.Get_size()} processes",
    flush=True
)

Overwriting mpi_test.py


In [25]:
!mpiexec --allow-run-as-root -n 4 python mpi_test.py

In [26]:
import os

print("File exists:", os.path.exists("mpi_test.py"))

if os.path.exists("mpi_test.py"):
    print(open("mpi_test.py").read())

File exists: True
from mpi4py import MPI

comm = MPI.COMM_WORLD

print(
    f"Hello from rank {comm.Get_rank()} "
    f"of {comm.Get_size()} processes",
    flush=True
)



In [27]:
!timeout 30s mpiexec --allow-run-as-root --oversubscribe -n 4 python -u mpi_test.py

In [28]:
import sys
import mpi4py
from mpi4py import MPI

print("Python:", sys.executable)
print("mpi4py:", mpi4py.__version__)
print("MPI library:", MPI.Get_library_version())
print("MPI size:", MPI.COMM_WORLD.Get_size())
print("MPI rank:", MPI.COMM_WORLD.Get_rank())

Python: /usr/bin/python3
mpi4py: 4.1.2
MPI library: Open MPI v4.1.6, package: Debian OpenMPI, ident: 4.1.6, repo rev: v4.1.6, Sep 30, 2023 
MPI size: 1
MPI rank: 0


In [29]:
!timeout 15s mpiexec --allow-run-as-root --oversubscribe --tag-output -n 4 python -u mpi_test.py
!echo "Exit code: $?"

Exit code: 0


In [30]:
!timeout 15s mpiexec --allow-run-as-root --oversubscribe --tag-output -n 4 /usr/bin/python3 -u mpi_test.py

In [31]:
import subprocess

result = subprocess.run(
    ["mpiexec", "--allow-run-as-root", "--oversubscribe",
     "-n", "4", "/usr/bin/python3", "-c",
     "from mpi4py import MPI; print('RANK', MPI.COMM_WORLD.Get_rank(), 'SIZE', MPI.COMM_WORLD.Get_size(), flush=True)"],
    capture_output=True,
    text=True,
    timeout=20
)

print("Return code:", result.returncode)
print("STDOUT:", repr(result.stdout))
print("STDERR:", repr(result.stderr))

Return code: 1
STDOUT: ''
STDERR: ''


In [32]:
!mpiexec --allow-run-as-root --oversubscribe -n 4 /bin/hostname

In [33]:
!mpiexec --allow-run-as-root --oversubscribe -n 4 /bin/echo MPI_WORKS

In [34]:
!ompi_info --version
!ompi_info --param plm all

Open MPI v4.1.6

http://www.open-mpi.org/community/help/
                 MCA plm: rsh (MCA v2.1.0, API v2.0.0, Component v4.1.6)
                 MCA plm: isolated (MCA v2.1.0, API v2.0.0, Component v4.1.6)
                 MCA plm: slurm (MCA v2.1.0, API v2.0.0, Component v4.1.6)


In [35]:
!timeout 10s mpiexec --allow-run-as-root --oversubscribe -n 2 /bin/echo TEST

In [36]:
import subprocess

for command in [
    ["mpiexec", "--version"],
    ["mpiexec", "--allow-run-as-root", "--oversubscribe",
     "-n", "2", "/bin/echo", "TEST"]
]:
    try:
        p = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=15
        )
        print("\nCommand:", command)
        print("Return code:", p.returncode)
        print("Output:", repr(p.stdout))
        print("Error:", repr(p.stderr))
    except subprocess.TimeoutExpired:
        print("\nCommand timed out:", command)


Command: ['mpiexec', '--version']
Return code: 0
Output: 'mpiexec (OpenRTE) 4.1.6\n\nReport bugs to http://www.open-mpi.org/community/help/\n'
Error: ''

Command: ['mpiexec', '--allow-run-as-root', '--oversubscribe', '-n', '2', '/bin/echo', 'TEST']
Return code: 1
Output: ''
Error: ''


In [37]:
import os
import subprocess

env = os.environ.copy()
env["OMPI_MCA_btl"] = "self,tcp"
env["OMPI_MCA_pml"] = "ob1"
env["OMPI_MCA_rmaps_base_oversubscribe"] = "1"

command = [
    "mpiexec",
    "--allow-run-as-root",
    "--oversubscribe",
    "-n", "4",
    "/usr/bin/python3",
    "-c",
    "from mpi4py import MPI; print('Rank', MPI.COMM_WORLD.Get_rank(), 'of', MPI.COMM_WORLD.Get_size(), flush=True)"
]

try:
    result = subprocess.run(
        command,
        env=env,
        capture_output=True,
        text=True,
        timeout=30
    )
    print("Return code:", result.returncode)
    print("STDOUT:\n", result.stdout)
    print("STDERR:\n", result.stderr)
except subprocess.TimeoutExpired:
    print("MPI launch timed out after 30 seconds.")

Return code: 0
STDOUT:
 Rank 2 of 4
Rank 1 of 4
Rank 0 of 4
Rank 3 of 4

STDERR:
 


In [38]:
import os

os.environ["OMPI_MCA_btl"] = "self,tcp"
os.environ["OMPI_MCA_pml"] = "ob1"
os.environ["OMPI_MCA_rmaps_base_oversubscribe"] = "1"

print("OpenMPI configuration saved for this notebook.")

OpenMPI configuration saved for this notebook.


In [39]:
%%writefile mpi_cuda_test.py
from mpi4py import MPI
from numba import cuda

comm = MPI.COMM_WORLD
rank = comm.Get_rank()

cuda.select_device(0)

print(
    f"Rank {rank}/{comm.Get_size()} | "
    f"GPU available: {cuda.is_available()}",
    flush=True
)

Writing mpi_cuda_test.py


In [40]:
!OMPI_MCA_btl=self,tcp OMPI_MCA_pml=ob1 mpiexec --allow-run-as-root --oversubscribe -n 4 /usr/bin/python3 -u mpi_cuda_test.py

In [41]:
import os
import subprocess

env = os.environ.copy()
env["OMPI_MCA_btl"] = "self,tcp"
env["OMPI_MCA_pml"] = "ob1"
env["OMPI_MCA_rmaps_base_oversubscribe"] = "1"

try:
    result = subprocess.run(
        [
            "mpiexec",
            "--allow-run-as-root",
            "--oversubscribe",
            "-n", "4",
            "/usr/bin/python3",
            "-u",
            "mpi_cuda_test.py"
        ],
        env=env,
        capture_output=True,
        text=True,
        timeout=60
    )

    print("Return code:", result.returncode)
    print("OUTPUT:")
    print(result.stdout)
    print("ERRORS:")
    print(result.stderr)

except subprocess.TimeoutExpired:
    print("MPI CUDA test timed out.")

Return code: 0
OUTPUT:
Rank 0/4 | GPU available: True
Rank 1/4 | GPU available: True
Rank 2/4 | GPU available: True
Rank 3/4 | GPU available: True

ERRORS:



In [42]:
%%writefile mpi_topology_test.py
from mpi4py import MPI

comm = MPI.COMM_WORLD

dims = [2, 2]
periods = [False, False]

cart = comm.Create_cart(
    dims=dims,
    periods=periods,
    reorder=False
)

rank = cart.Get_rank()
coords = cart.Get_coords(rank)

up, down = cart.Shift(0, 1)
left, right = cart.Shift(1, 1)

print(
    f"Rank {rank}: coordinates={coords}, "
    f"up={up}, down={down}, "
    f"left={left}, right={right}",
    flush=True
)

Writing mpi_topology_test.py


In [43]:
import os
import subprocess

env = os.environ.copy()
env["OMPI_MCA_btl"] = "self,tcp"
env["OMPI_MCA_pml"] = "ob1"

result = subprocess.run(
    [
        "mpiexec", "--allow-run-as-root",
        "--oversubscribe", "-n", "4",
        "/usr/bin/python3", "-u",
        "mpi_topology_test.py"
    ],
    env=env,
    capture_output=True,
    text=True,
    timeout=60
)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)

Return code: 0
Rank 1: coordinates=[0, 1], up=-2, down=3, left=0, right=-2
Rank 2: coordinates=[1, 0], up=0, down=-2, left=-2, right=3
Rank 0: coordinates=[0, 0], up=-2, down=2, left=-2, right=1
Rank 3: coordinates=[1, 1], up=1, down=-2, left=2, right=-2




In [44]:
%%writefile task2_test.py
import numpy as np
from mpi4py import MPI
from distributed_cuda_solver import (
    init_distributed_context,
    initialize_subdomains
)

username = "belkamenbirgee"

(
    cart, rank, size, row, col,
    up, down, left, right, seed
) = init_distributed_context(username)

u, kappa, source = initialize_subdomains(
    row, col, seed,
    up, down, left, right
)

print(f"\nRANK {rank}")
print("Coordinates:", (row, col))
print("u shape:", u.shape)
print("kappa shape:", kappa.shape)
print("source shape:", source.shape)
print("Kappa range:", kappa.min(), kappa.max())
print("All values finite:", np.isfinite(u).all()
      and np.isfinite(kappa).all()
      and np.isfinite(source).all())

print("Top boundary zero:", np.all(u[0, :] == 0)
      if up == MPI.PROC_NULL else "Has neighbor")

print("Bottom boundary zero:", np.all(u[-1, :] == 0)
      if down == MPI.PROC_NULL else "Has neighbor")

print("Left boundary zero:", np.all(u[:, 0] == 0)
      if left == MPI.PROC_NULL else "Has neighbor")

print("Right boundary zero:", np.all(u[:, -1] == 0)
      if right == MPI.PROC_NULL else "Has neighbor")

cart.Barrier()

Writing task2_test.py


In [45]:
import os
import subprocess

env = os.environ.copy()
env["OMPI_MCA_btl"] = "self,tcp"
env["OMPI_MCA_pml"] = "ob1"
env["OMPI_MCA_rmaps_base_oversubscribe"] = "1"

result = subprocess.run(
    [
        "mpiexec", "--allow-run-as-root",
        "--oversubscribe", "-n", "4",
        "/usr/bin/python3", "-u", "task2_test.py"
    ],
    env=env,
    capture_output=True,
    text=True,
    timeout=60
)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)

Return code: 0
Rank 0: coordinates=[0, 0], neighbors=(-2, 2, -2, 1)
Rank 2: coordinates=[1, 0], neighbors=(0, -2, -2, 3)
Rank 1: coordinates=[0, 1], neighbors=(-2, 3, 0, -2)

RANK 1
Coordinates: (0, 1)
u shape: (258, 258)
kappa shape: (258, 258)
source shape: (258, 258)
Kappa range: 0.08000037502884799 0.2399981248663085
All values finite: True
Top boundary zero: True
Bottom boundary zero: Has neighbor
Left boundary zero: Has neighbor
Right boundary zero: True

RANK 0
Coordinates: (0, 0)
u shape: (258, 258)
kappa shape: (258, 258)
source shape: (258, 258)
Kappa range: 0.0800018751336915 0.23999962497115201
All values finite: True
Top boundary zero:
RANK 2
Coordinates: (1, 0)
u shape: (258, 258)
kappa shape: (258, 258)
source shape: (258, 258)
Kappa range: 0.0800018751336915 0.23999962497115201
All values finite: True
Top boundary zero: Has neighbor
Bottom boundary zero: True
Left boundary zero: True
Right boundary zero: Has neighbor
 True
Bottom boundary zero: Has neighbor
Left boundar

In [46]:
%%writefile task3_test.py
import numpy as np
from mpi4py import MPI
from numba import cuda

from distributed_cuda_solver import (
    init_distributed_context,
    initialize_subdomains,
    exchange_halos_async
)

username = "belkamenbirgee"

(
    cart, rank, size, row, col,
    up, down, left, right, seed
) = init_distributed_context(username)

u_host, kappa_host, source_host = initialize_subdomains(
    row, col, seed, up, down, left, right
)

u = cuda.to_device(u_host)
stream = cuda.stream()

# Perform GPU packing, MPI exchange, and GPU unpacking
exchange_halos_async(
    u, cart, up, down, left, right, stream
)

stream.synchronize()
result = u.copy_to_host()

# Verify received ghost cells against neighboring interior edges.
checks = []

for neighbor, send_edge, ghost_edge, tag in [
    (up, result[1, 1:257], result[0, 1:257], 0),
    (down, result[256, 1:257], result[257, 1:257], 1),
    (left, result[1:257, 1], result[1:257, 0], 2),
    (right, result[1:257, 256], result[1:257, 257], 3)
]:
    if neighbor == MPI.PROC_NULL:
        checks.append(np.all(ghost_edge == 0.0))
    else:
        expected = np.empty_like(send_edge)
        cart.Sendrecv(
            sendbuf=send_edge.copy(),
            dest=neighbor,
            sendtag=tag,
            recvbuf=expected,
            source=neighbor,
            recvtag={0: 1, 1: 0, 2: 3, 3: 2}[tag]
        )
        checks.append(np.allclose(ghost_edge, expected))

passed = all(checks)

print(
    f"Rank {rank}: halo checks={checks}, "
    f"PASS={passed}",
    flush=True
)

all_passed = cart.allreduce(
    int(passed), op=MPI.SUM
)

if rank == 0:
    print(f"\nRanks passing halo test: {all_passed}/4")

cart.Barrier()

Writing task3_test.py


In [47]:
import os
import subprocess

env = os.environ.copy()
env["OMPI_MCA_btl"] = "self,tcp"
env["OMPI_MCA_pml"] = "ob1"
env["OMPI_MCA_rmaps_base_oversubscribe"] = "1"

result = subprocess.run(
    [
        "mpiexec", "--allow-run-as-root",
        "--oversubscribe", "-n", "4",
        "/usr/bin/python3", "-u", "task3_test.py"
    ],
    env=env,
    capture_output=True,
    text=True,
    timeout=120
)

print("Return code:", result.returncode)
print("OUTPUT:")
print(result.stdout)
print("ERRORS:")
print(result.stderr)

Return code: 0
OUTPUT:
Rank 0: coordinates=[0, 0], neighbors=(-2, 2, -2, 1)
Rank 1: coordinates=[0, 1], neighbors=(-2, 3, 0, -2)
Rank 2: coordinates=[1, 0], neighbors=(0, -2, -2, 3)
Rank 3: coordinates=[1, 1], neighbors=(1, -2, 2, -2)
Rank 0: halo checks=[np.True_, True, np.True_, True], PASS=True
Rank 2: halo checks=[True, np.True_, np.True_, True], PASS=True
Rank 1: halo checks=[np.True_, True, True, np.True_], PASS=True
Rank 3: halo checks=[True, np.True_, True, np.True_], PASS=True

Ranks passing halo test: 4/4

ERRORS:
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib

In [48]:
%%writefile task4_test.py
import numpy as np
from mpi4py import MPI
from numba import cuda

from distributed_cuda_solver import (
    init_distributed_context,
    initialize_subdomains,
    tiled_stencil_kernel,
    DX,
    DY
)

username = "belkamenbirgee"

(
    cart, rank, size, row, col,
    up, down, left, right, seed
) = init_distributed_context(username)

u_host, kappa_host, source_host = initialize_subdomains(
    row, col, seed, up, down, left, right
)

u = cuda.to_device(u_host)
kappa = cuda.to_device(kappa_host)
source = cuda.to_device(source_host)

out = cuda.to_device(np.zeros_like(u_host))

kappa_max = np.max(kappa_host)
dt_max = DX**2 / (4.0 * kappa_max)
dt = 0.2 * dt_max

# Launch the active shared-memory stencil kernel
tiled_stencil_kernel[(16, 16), (16, 16)](
    u, kappa, source, out, dt,
    1, 256, 1, 256
)

cuda.synchronize()

result = out.copy_to_host()

valid = np.isfinite(result[1:257, 1:257]).all()

print(
    f"Rank {rank}: stencil executed | "
    f"finite output={valid} | "
    f"min={result[1:257, 1:257].min():.8f} | "
    f"max={result[1:257, 1:257].max():.8f}",
    flush=True
)

passed = cart.allreduce(int(valid), op=MPI.SUM)

if rank == 0:
    print(f"\nRanks passing stencil test: {passed}/4")

cart.Barrier()

Writing task4_test.py


In [49]:
import os
import subprocess

env = os.environ.copy()
env["OMPI_MCA_btl"] = "self,tcp"
env["OMPI_MCA_pml"] = "ob1"
env["OMPI_MCA_rmaps_base_oversubscribe"] = "1"

result = subprocess.run(
    [
        "mpiexec", "--allow-run-as-root",
        "--oversubscribe", "-n", "4",
        "/usr/bin/python3", "-u", "task4_test.py"
    ],
    env=env,
    capture_output=True,
    text=True,
    timeout=120
)

print("Return code:", result.returncode)
print("OUTPUT:")
print(result.stdout)
print("ERRORS:")
print(result.stderr)

Return code: 0
OUTPUT:
Rank 0: coordinates=[0, 0], neighbors=(-2, 2, -2, 1)
Rank 2: coordinates=[1, 0], neighbors=(0, -2, -2, 3)
Rank 3: coordinates=[1, 1], neighbors=(1, -2, 2, -2)Rank 1: coordinates=[0, 1], neighbors=(-2, 3, 0, -2)

Rank 3: stencil executed | finite output=True | min=0.00000000 | max=0.82316513
Rank 0: stencil executed | finite output=True | min=0.00000001 | max=1.24013470
Rank 1: stencil executed | finite output=True | min=0.00000007 | max=0.60487767
Rank 2: stencil executed | finite output=True | min=0.00000000 | max=1.03022615

Ranks passing stencil test: 4/4

ERRORS:



In [50]:
from pathlib import Path

file_path = Path("distributed_cuda_solver.py")
content = file_path.read_text()

for i, line in enumerate(content.splitlines(), start=1):
    if "TASK 4" in line or "TASK 5" in line:
        print(f"Line {i}: {line}")

Line 421: # TASK 4: 2D SHARED-MEMORY TILED CUDA STENCIL
Line 516: # TASK 4: KERNEL LAUNCH CONFIGURATION
Line 555: # TASK 4: SHARED-MEMORY TILED CUDA STENCIL
Line 637: # TASK 4: TIME-STEP ORCHESTRATOR
Line 675: # TASK 5: ENERGY, RESIDUAL, PTX AUDIT AND SOLVER


In [51]:
from pathlib import Path

file_path = Path("distributed_cuda_solver.py")

# Read the existing solver
content = file_path.read_text()

# Keep everything before the first Task 4 section
marker = "# TASK 4: 2D SHARED-MEMORY TILED CUDA STENCIL"
position = content.find(marker)

if position == -1:
    raise ValueError("Task 4 marker was not found.")

# Create a backup
backup_path = Path("distributed_cuda_solver_backup.py")
backup_path.write_text(content)

# Preserve Tasks 1–3
task123 = content[:position]

print("Backup created:", backup_path.exists())
print("Tasks 1–3 preserved:", len(task123.splitlines()), "lines")
print("Old Task 4 and Task 5 sections located for replacement.")

Backup created: True
Tasks 1–3 preserved: 420 lines
Old Task 4 and Task 5 sections located for replacement.


In [52]:
from pathlib import Path

content = Path("distributed_cuda_solver.py").read_text()
task123 = content[:content.find("# TASK 4: 2D SHARED-MEMORY TILED CUDA STENCIL")]

for keyword in [
    "def pack_halos_kernel",
    "def unpack_halos_kernel",
    "def exchange_halos_async",
    "def initialize_subdomains",
    "def init_distributed_context"
]:
    print("\n" + "=" * 60)
    print(keyword)
    start = task123.find(keyword)
    if start == -1:
        print("NOT FOUND")
        continue

    end = task123.find("\ndef ", start + 1)
    if end == -1:
        end = len(task123)

    print(task123[start:end])


def pack_halos_kernel
def pack_halos_kernel(u, send_buffer, direction, nx, ny):

    idx = cuda.grid(1)

    if idx < max(nx, ny):

        if direction == 0 and idx < nx:
            send_buffer[idx] = u[1, idx + 1]

        elif direction == 1 and idx < nx:
            send_buffer[idx] = u[ny, idx + 1]

        elif direction == 2 and idx < ny:
            send_buffer[idx] = u[idx + 1, 1]

        elif direction == 3 and idx < ny:
            send_buffer[idx] = u[idx + 1, nx]


@cuda.jit

def unpack_halos_kernel
def unpack_halos_kernel(u, recv_buffer, direction, nx, ny):

    idx = cuda.grid(1)

    if idx < max(nx, ny):

        if direction == 0 and idx < nx:
            u[0, idx + 1] = recv_buffer[idx]

        elif direction == 1 and idx < nx:
            u[ny + 1, idx + 1] = recv_buffer[idx]

        elif direction == 2 and idx < ny:
            u[idx + 1, 0] = recv_buffer[idx]

        elif direction == 3 and idx < ny:
            u[idx + 1, nx + 1] = recv_buffer[idx]


# ====

In [53]:

from pathlib import Path

file_path = Path("distributed_cuda_solver.py")
content = file_path.read_text()

marker = "# TASK 4: 2D SHARED-MEMORY TILED CUDA STENCIL"
position = content.find(marker)

if position == -1:
    raise ValueError("Task 4 marker not found.")

# Preserve the working Tasks 1–3
task123 = content[:position]

replacement = r'''
# ============================================================
# TASK 4: SHARED-MEMORY TILED CUDA STENCIL
# ============================================================

@cuda.jit
def tiled_stencil_kernel(u, kappa, source, out, dt,
                         boundary_only):
    tile = cuda.shared.array((18, 18), dtype=float64)

    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y

    tid = ty * 16 + tx

    base_r = cuda.blockIdx.y * 16
    base_c = cuda.blockIdx.x * 16

    # Cooperatively load all 324 values into shared memory.
    # 256 threads load one value each; first 68 load a second.
    for item in range(2):
        index = tid + item * 256

        if index < 324:
            sr = index // 18
            sc = index % 18

            tile[sr, sc] = u[base_r + sr, base_c + sc]

    cuda.syncthreads()

    r = base_r + ty + 1
    c = base_c + tx + 1

    if r <= 256 and c <= 256:

        is_deep = (
            r >= 2 and r <= 255
            and c >= 2 and c <= 255
        )

        # Deep-interior pass skips the boundary ring.
        if boundary_only == 0 or not is_deep:

            center = tile[ty + 1, tx + 1]
            north = tile[ty, tx + 1]
            south = tile[ty + 2, tx + 1]
            west = tile[ty + 1, tx]
            east = tile[ty + 1, tx + 2]

            kc = kappa[r, c]

            kn = 0.5 * (kc + kappa[r - 1, c])
            ks = 0.5 * (kc + kappa[r + 1, c])
            kw = 0.5 * (kc + kappa[r, c - 1])
            ke = 0.5 * (kc + kappa[r, c + 1])

            dx = 1.0 / 513.0
            dy = 1.0 / 513.0

            diffusion = (
                (ke * (east - center)
                 - kw * (center - west)) / (dx * dx)
                +
                (ks * (south - center)
                 - kn * (center - north)) / (dy * dy)
            )

            reaction = -0.15 * center * center * center

            out[r, c] = center + dt * (
                diffusion + reaction + source[r, c]
            )


# ============================================================
# TASK 4: ASYNCHRONOUS HALO EXCHANGE
# ============================================================

def begin_halo_exchange(u, cart_comm,
                        nbr_up, nbr_down,
                        nbr_left, nbr_right,
                        stream_boundary):

    nx = LOCAL_NX
    ny = LOCAL_NY

    neighbors = [
        nbr_up, nbr_down, nbr_left, nbr_right
    ]

    lengths = [nx, nx, ny, ny]
    send_tags = [0, 1, 2, 3]
    recv_tags = [1, 0, 3, 2]

    max_length = max(nx, ny)
    threads = 256
    blocks = (max_length + threads - 1) // threads

    host_send = [
        cuda.pinned_array(max_length, dtype=np.float64)
        for _ in range(4)
    ]

    host_recv = [
        cuda.pinned_array(max_length, dtype=np.float64)
        for _ in range(4)
    ]

    device_send = [
        cuda.device_array(max_length, dtype=np.float64)
        for _ in range(4)
    ]

    device_recv = [
        cuda.device_array(max_length, dtype=np.float64)
        for _ in range(4)
    ]

    # GPU packing and D2H transfer.
    for direction in range(4):
        pack_halos_kernel[blocks, threads, stream_boundary](
            u, device_send[direction], direction, nx, ny
        )

        device_send[direction].copy_to_host(
            host_send[direction],
            stream=stream_boundary
        )

    # MPI must not read the send buffers until D2H is complete.
    stream_boundary.synchronize()

    for direction in range(4):
        host_recv[direction][:] = 0.0

    requests = []

    # Post all receives first.
    for direction in range(4):
        requests.append(
            cart_comm.Irecv(
                host_recv[direction][:lengths[direction]],
                source=neighbors[direction],
                tag=recv_tags[direction]
            )
        )

    # Then post all sends.
    for direction in range(4):
        requests.append(
            cart_comm.Isend(
                host_send[direction][:lengths[direction]],
                dest=neighbors[direction],
                tag=send_tags[direction]
            )
        )

    # Do NOT wait here. The caller computes the deep interior.

    return {
        "requests": requests,
        "host_send": host_send,
        "host_recv": host_recv,
        "device_recv": device_recv,
        "lengths": lengths,
        "blocks": blocks,
        "threads": threads,
        "u": u
    }


def finish_halo_exchange(exchange, interior_stream,
                         stream_boundary):

    # Wait for MPI while deep-interior CUDA work can run.
    MPI.Request.Waitall(exchange["requests"])

    # Ensure deep computation has finished before ghost updates.
    interior_stream.synchronize()

    u = exchange["u"]

    # Copy received halos to GPU and unpack.
    for direction in range(4):
        exchange["device_recv"][direction].copy_to_device(
            exchange["host_recv"][direction],
            stream=stream_boundary
        )

        unpack_halos_kernel[
            exchange["blocks"],
            exchange["threads"],
            stream_boundary
        ](
            u,
            exchange["device_recv"][direction],
            direction,
            LOCAL_NX,
            LOCAL_NY
        )

    stream_boundary.synchronize()


# ============================================================
# TASK 5: METRICS AND PTX AUDIT
# ============================================================

def calculate_metrics(u, previous_u):
    interior = u[1:-1, 1:-1]
    previous = previous_u[1:-1, 1:-1]

    energy = float(np.sum(interior) * DX * DY)

    residual = float(
        np.sqrt(np.sum((interior - previous) ** 2))
    )

    return energy, residual


def global_metrics(cart_comm, d_u, d_previous):
    u_host = d_u.copy_to_host()
    previous_host = d_previous.copy_to_host()

    local_energy, local_residual = calculate_metrics(
        u_host, previous_host
    )

    energy = cart_comm.allreduce(
        local_energy, op=MPI.SUM
    )

    residual_squared = cart_comm.allreduce(
        local_residual ** 2, op=MPI.SUM
    )

    return energy, float(np.sqrt(residual_squared))


def ptx_audit():
    compiled = tiled_stencil_kernel.inspect_asm()

    if isinstance(compiled, dict):
        ptx = next(iter(compiled.values()))
    else:
        ptx = compiled

    return {
        "shared_loads": ptx.count("ld.shared"),
        "shared_stores": ptx.count("st.shared"),
        "bar_sync": ptx.count("bar.sync"),
        "has_shared_memory": (
            "ld.shared" in ptx and "st.shared" in ptx
        ),
        "has_barrier": "bar.sync" in ptx
    }


# ============================================================
# TASK 5: ONE SERIALIZED OR OVERLAPPED SIMULATION
# ============================================================

def run_simulation(initial_u, kappa, source,
                   cart_comm, neighbors, dt,
                   mode, rank):

    d_u = cuda.to_device(initial_u)
    d_next = cuda.device_array_like(d_u)
    d_previous = cuda.device_array_like(d_u)

    stream_boundary = cuda.stream()
    stream_interior = cuda.stream()

    nbr_up, nbr_down, nbr_left, nbr_right = neighbors

    blocks = (16, 16)
    threads = (16, 16)

    d_kappa = cuda.to_device(kappa)
    d_source = cuda.to_device(source)

    energy_trace = []
    residual_trace = []

    # Initial metric at step 0.
    d_previous.copy_to_device(d_u)
    energy, residual = global_metrics(
        cart_comm, d_u, d_previous
    )

    if rank == 0:
        energy_trace.append((0, energy))
        residual_trace.append((0, residual))

    cart_comm.Barrier()
    start_time = MPI.Wtime()

    for step in range(1, STEPS + 1):

        # Preserve the previous state for residual calculation.
        d_previous.copy_to_device(d_u)

        # Keep ghost cells from the current state.
        d_next.copy_to_device(d_u)

        if mode == "serialized":

            # Complete halo exchange before stencil computation.
            exchange = begin_halo_exchange(
                d_u, cart_comm,
                nbr_up, nbr_down,
                nbr_left, nbr_right,
                stream_boundary
            )

            finish_halo_exchange(
                exchange, stream_interior, stream_boundary
            )

            tiled_stencil_kernel[
                blocks, threads, stream_boundary
            ](
                d_u, d_kappa, d_source, d_next,
                dt, 0
            )

            stream_boundary.synchronize()

        else:

            # Start nonblocking halo communication.
            exchange = begin_halo_exchange(
                d_u, cart_comm,
                nbr_up, nbr_down,
                nbr_left, nbr_right,
                stream_boundary
            )

            # Compute only the deep interior while MPI is active.
            tiled_stencil_kernel[
                blocks, threads, stream_interior
            ](
                d_u, d_kappa, d_source, d_next,
                dt, 1
            )

            # Finish communication and unpack the ghost cells.
            finish_halo_exchange(
                exchange, stream_interior, stream_boundary
            )

            # Compute only the boundary ring.
            tiled_stencil_kernel[
                blocks, threads, stream_boundary
            ](
                d_u, d_kappa, d_source, d_next,
                dt, 2
            )

            stream_boundary.synchronize()

        d_u, d_next = d_next, d_u

        # Global metrics every 100 steps.
        if step % 100 == 0:
            energy, residual = global_metrics(
                cart_comm, d_u, d_previous
            )

            if rank == 0:
                energy_trace.append((step, energy))
                residual_trace.append((step, residual))

                print(
                    f"{mode} | Step {step:3d} | "
                    f"Energy={energy:.12e} | "
                    f"L2 residual={residual:.12e}"
                )

    stream_boundary.synchronize()
    stream_interior.synchronize()

    elapsed = MPI.Wtime() - start_time
    elapsed = cart_comm.allreduce(elapsed, op=MPI.MAX)

    final_u = d_u.copy_to_host()

    return final_u, elapsed, energy_trace, residual_trace


# ============================================================
# TASK 5: COMPLETE SOLVER
# ============================================================

def run_solver(github_user):

    (
        cart_comm, rank, size,
        row_rank, col_rank,
        nbr_up, nbr_down, nbr_left, nbr_right,
        instance_seed
    ) = init_distributed_context(github_user)

    u, kappa, source = initialize_subdomains(
        row_rank, col_rank, instance_seed,
        nbr_up, nbr_down, nbr_left, nbr_right
    )

    # Use a globally consistent maximum diffusivity.
    local_kappa_max = float(np.max(kappa))
    kappa_max = cart_comm.allreduce(
        local_kappa_max, op=MPI.MAX
    )

    dt_max = (DX ** 2) / (4.0 * kappa_max)
    dt = 0.2 * dt_max

    if rank == 0:
        print("\\nStarting distributed CUDA stencil solver")
        print("MPI ranks:", size)
        print("Global grid:", GLOBAL_NX, "x", GLOBAL_NY)
        print("Local grid:", LOCAL_NX, "x", LOCAL_NY)
        print("Steps:", STEPS)
        print("Maximum diffusivity:", kappa_max)
        print("dt:", dt)

    neighbors = (
        nbr_up, nbr_down, nbr_left, nbr_right
    )

    initial_u = u.copy()

    # Reference run: serialized communication and computation.
    serialized, serial_time, serial_energy, serial_residual = (
        run_simulation(
            initial_u, kappa, source,
            cart_comm, neighbors, dt,
            "serialized", rank
        )
    )

    cart_comm.Barrier()

    # Test run: communication overlaps deep-interior computation.
    overlapped, overlap_time, overlap_energy, overlap_residual = (
        run_simulation(
            initial_u, kappa, source,
            cart_comm, neighbors, dt,
            "overlapped", rank
        )
    )

    # Compare final states across all ranks.
    local_max_diff = float(
        np.max(np.abs(serialized - overlapped))
    )

    max_diff = cart_comm.allreduce(
        local_max_diff, op=MPI.MAX
    )

    audit = ptx_audit()

    if rank == 0:
        print("\\n========== FINAL RESULTS ==========")
        print("Serialized time:", serial_time)
        print("Overlapped time:", overlap_time)
        print("Maximum final-state difference:", max_diff)
        print("Numerical comparison (< 1e-12):", max_diff < 1e-12)
        print("PTX audit:", audit)

        manifest = {
            "github_username": github_user,
            "instance_seed": int(instance_seed),
            "gpu_uuid": get_gpu_uuid(),
            "mpi_ranks": int(size),
            "global_grid": [GLOBAL_NX, GLOBAL_NY],
            "local_grid": [LOCAL_NX, LOCAL_NY],
            "steps": STEPS,
            "dt": float(dt),
            "kappa_max": float(kappa_max),
            "serialized_time_seconds": float(serial_time),
            "overlapped_time_seconds": float(overlap_time),
            "max_final_state_difference": float(max_diff),
            "numerical_match_1e_12": bool(max_diff < 1e-12),
            "ptx_audit": audit,
            "energy_trace": serial_energy
        }

        with open("submission_manifest.json", "w") as f:
            json.dump(manifest, f, indent=4)

        print("\\nCreated submission_manifest.json")

    cart_comm.Barrier()
    cart_comm.Free()

    return max_diff


if __name__ == "__main__":
    if len(sys.argv) < 2:
        raise RuntimeError(
            "Usage: python distributed_cuda_solver.py GITHUB_USERNAME"
        )

    run_solver(sys.argv[1])
'''

file_path.write_text(task123 + replacement)

print("Replacement complete.")
print("Preserved Tasks 1–3:", len(task123.splitlines()), "lines")
print("Total file lines:", len(file_path.read_text().splitlines()))


Replacement complete.
Preserved Tasks 1–3: 420 lines
Total file lines: 929


In [54]:
import py_compile

py_compile.compile(
    "distributed_cuda_solver.py",
    doraise=True
)

print("SUCCESS: No Python syntax errors.")

SUCCESS: No Python syntax errors.


In [55]:
import os
import subprocess

env = os.environ.copy()
env["OMPI_MCA_btl"] = "self,tcp"
env["OMPI_MCA_pml"] = "ob1"
env["OMPI_MCA_rmaps_base_oversubscribe"] = "1"

test_code = r'''
from mpi4py import MPI
from numba import cuda

comm = MPI.COMM_WORLD
rank = comm.Get_rank()

cuda.select_device(0)

print(
    f"Rank {rank}/{comm.Get_size()} | "
    f"GPU available: {cuda.is_available()} | "
    f"Device: {cuda.get_current_device().name.decode()}",
    flush=True
)
'''

result = subprocess.run(
    [
        "/usr/bin/python3",
        "-c", test_code
    ],
    env=env,
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print("OUTPUT:")
print(result.stdout)

if result.stderr:
    print("ERRORS:")
    print(result.stderr)

Return code: 1
OUTPUT:

ERRORS:
Traceback (most recent call last):
  File "<string>", line 13, in <module>
    f"Device: {cuda.get_current_device().name.decode()}",
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'str' object has no attribute 'decode'. Did you mean: 'encode'?



In [56]:
import os
import subprocess

env = os.environ.copy()
env["OMPI_MCA_btl"] = "self,tcp"
env["OMPI_MCA_pml"] = "ob1"
env["OMPI_MCA_rmaps_base_oversubscribe"] = "1"

test_code = r'''
from mpi4py import MPI
from numba import cuda

comm = MPI.COMM_WORLD
rank = comm.Get_rank()

cuda.select_device(0)

print(
    f"Rank {rank}/{comm.Get_size()} | "
    f"GPU available: {cuda.is_available()} | "
    f"Device: {cuda.get_current_device().name}",
    flush=True
)
'''

result = subprocess.run(
    [
        "mpiexec",
        "--allow-run-as-root",
        "--oversubscribe",
        "-n", "4",
        "/usr/bin/python3",
        "-c", test_code
    ],
    env=env,
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print("OUTPUT:")
print(result.stdout)

if result.stderr:
    print("ERRORS:")
    print(result.stderr)

Return code: 0
OUTPUT:
Rank 3/4 | GPU available: True | Device: Tesla T4
Rank 1/4 | GPU available: True | Device: Tesla T4
Rank 0/4 | GPU available: True | Device: Tesla T4
Rank 2/4 | GPU available: True | Device: Tesla T4



In [57]:
import os
import subprocess

env = os.environ.copy()
env["OMPI_MCA_btl"] = "self,tcp"
env["OMPI_MCA_pml"] = "ob1"
env["OMPI_MCA_rmaps_base_oversubscribe"] = "1"

test_code = r'''
import numpy as np
from mpi4py import MPI
from numba import cuda

from distributed_cuda_solver import (
    tiled_stencil_kernel,
    LOCAL_NX,
    LOCAL_NY
)

comm = MPI.COMM_WORLD
rank = comm.Get_rank()

cuda.select_device(0)

u = np.ones((258, 258), dtype=np.float64)
kappa = np.full((258, 258), 0.15, dtype=np.float64)
source = np.zeros((258, 258), dtype=np.float64)
out = np.zeros((258, 258), dtype=np.float64)

d_u = cuda.to_device(u)
d_k = cuda.to_device(kappa)
d_s = cuda.to_device(source)
d_out = cuda.to_device(out)

tiled_stencil_kernel[(16, 16), (16, 16)](
    d_u, d_k, d_s, d_out, 0.000001, 0
)

result = d_out.copy_to_host()

valid = np.isfinite(result).all()
print(
    f"Rank {rank}: Kernel executed | "
    f"Finite output: {valid} | "
    f"Shape: {result.shape}",
    flush=True
)

assert valid
'''

result = subprocess.run(
    [
        "mpiexec",
        "--allow-run-as-root",
        "--oversubscribe",
        "-n", "4",
        "/usr/bin/python3",
        "-c", test_code
    ],
    env=env,
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print("OUTPUT:")
print(result.stdout)

if result.stderr:
    print("ERRORS:")
    print(result.stderr)

Return code: 0
OUTPUT:
Rank 0: Kernel executed | Finite output: True | Shape: (258, 258)
Rank 3: Kernel executed | Finite output: True | Shape: (258, 258)
Rank 1: Kernel executed | Finite output: True | Shape: (258, 258)
Rank 2: Kernel executed | Finite output: True | Shape: (258, 258)



In [58]:
import os
import subprocess

env = os.environ.copy()
env["OMPI_MCA_btl"] = "self,tcp"
env["OMPI_MCA_pml"] = "ob1"
env["OMPI_MCA_rmaps_base_oversubscribe"] = "1"

test_code = r'''
import numpy as np
from mpi4py import MPI
from numba import cuda

from distributed_cuda_solver import (
    begin_halo_exchange,
    finish_halo_exchange,
    tiled_stencil_kernel
)

comm = MPI.COMM_WORLD
rank = comm.Get_rank()

cart = comm.Create_cart(
    dims=[2, 2],
    periods=[False, False],
    reorder=False
)

up, down = cart.Shift(0, 1)
left, right = cart.Shift(1, 1)

cuda.select_device(0)

# Each rank fills its interior with its own rank number + 1.
u = np.zeros((258, 258), dtype=np.float64)
u[1:257, 1:257] = rank + 1

d_u = cuda.to_device(u)
d_k = cuda.to_device(np.full((258, 258), 0.15))
d_s = cuda.to_device(np.zeros((258, 258)))
d_out = cuda.device_array((258, 258), dtype=np.float64)

stream_boundary = cuda.stream()
stream_interior = cuda.stream()

# Start nonblocking communication.
exchange = begin_halo_exchange(
    d_u, cart, up, down, left, right,
    stream_boundary
)

# Perform deep-interior CUDA work while MPI is active.
tiled_stencil_kernel[(16, 16), (16, 16), stream_interior](
    d_u, d_k, d_s, d_out, 0.000001, 1
)

# Complete communication and unpack received halos.
finish_halo_exchange(
    exchange, stream_interior, stream_boundary
)

result = d_u.copy_to_host()

checks = []

# Verify each received ghost edge.
for direction, neighbor, edge in [
    ("up", up, result[0, 1:257]),
    ("down", down, result[257, 1:257]),
    ("left", left, result[1:257, 0]),
    ("right", right, result[1:257, 257])
]:
    expected = 0.0 if neighbor == MPI.PROC_NULL else float(neighbor + 1)
    passed = np.all(edge == expected)
    checks.append(passed)

    print(
        f"Rank {rank} | {direction}: "
        f"{'PASS' if passed else 'FAIL'}",
        flush=True
    )

assert all(checks), f"Rank {rank}: halo validation failed"

print(f"Rank {rank}: ALL HALO CHECKS PASSED", flush=True)

cart.Free()
'''

result = subprocess.run(
    [
        "mpiexec",
        "--allow-run-as-root",
        "--oversubscribe",
        "-n", "4",
        "/usr/bin/python3",
        "-c", test_code
    ],
    env=env,
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print("OUTPUT:")
print(result.stdout)

if result.stderr:
    print("ERRORS:")
    print(result.stderr)

Return code: 0
OUTPUT:
Rank 2 | up: PASS
Rank 2 | down: PASS
Rank 2 | left: PASS
Rank 2 | right: PASS
Rank 2: ALL HALO CHECKS PASSED
Rank 0 | up: PASS
Rank 0 | down: PASS
Rank 0 | left: PASS
Rank 0 | right: PASS
Rank 0: ALL HALO CHECKS PASSED
Rank 3 | up: PASS
Rank 3 | down: PASS
Rank 3 | left: PASS
Rank 3 | right: PASS
Rank 3: ALL HALO CHECKS PASSED
Rank 1 | up: PASS
Rank 1 | down: PASS
Rank 1 | left: PASS
Rank 1 | right: PASS
Rank 1: ALL HALO CHECKS PASSED

ERRORS:
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher

In [59]:
from pathlib import Path

path = Path("distributed_cuda_solver.py")
content = path.read_text()

old = """        # Deep-interior pass skips the boundary ring.
        if boundary_only == 0 or not is_deep:
"""

new = """        # 0 = full interior, 1 = deep interior, 2 = boundary ring.
        should_compute = (
            boundary_only == 0
            or (boundary_only == 1 and is_deep)
            or (boundary_only == 2 and not is_deep)
        )

        if should_compute:
"""

if old not in content:
    raise ValueError("Expected kernel condition was not found.")

path.write_text(content.replace(old, new, 1))

print("Kernel region selection corrected.")

Kernel region selection corrected.


In [60]:
import py_compile

py_compile.compile("distributed_cuda_solver.py", doraise=True)
print("SUCCESS: Syntax check passed.")

SUCCESS: Syntax check passed.


In [61]:
import os
import subprocess

env = os.environ.copy()
env["OMPI_MCA_btl"] = "self,tcp"
env["OMPI_MCA_pml"] = "ob1"
env["OMPI_MCA_rmaps_base_oversubscribe"] = "1"

code = r'''
import numpy as np
from mpi4py import MPI
from numba import cuda
from distributed_cuda_solver import tiled_stencil_kernel

rank = MPI.COMM_WORLD.Get_rank()
cuda.select_device(0)

u = cuda.to_device(np.ones((258,258), dtype=np.float64))
k = cuda.to_device(np.full((258,258), 0.15))
s = cuda.to_device(np.zeros((258,258)))
out = cuda.to_device(np.zeros((258,258), dtype=np.float64))

# Deep interior only
tiled_stencil_kernel[(16,16),(16,16)](
    u, k, s, out, 0.001, 1
)
deep = out.copy_to_host()

assert np.all(deep[2:256, 2:256] != 0)
assert np.all(deep[1, :] == 0)
assert np.all(deep[256, :] == 0)
assert np.all(deep[:, 1] == 0)
assert np.all(deep[:, 256] == 0)

# Boundary ring only
out = cuda.to_device(np.zeros((258,258), dtype=np.float64))
tiled_stencil_kernel[(16,16),(16,16)](
    u, k, s, out, 0.001, 2
)
boundary = out.copy_to_host()

assert np.all(boundary[2:256, 2:256] == 0)
assert np.any(boundary[1, 1:257] != 0)

print(f"Rank {rank}: deep interior and boundary ring PASS", flush=True)
'''

result = subprocess.run(
    ["mpiexec", "--allow-run-as-root", "--oversubscribe",
     "-n", "4", "/usr/bin/python3", "-c", code],
    env=env, capture_output=True, text=True
)

print("Return code:", result.returncode)
print(result.stdout)
if result.stderr:
    print("ERRORS:")
    print(result.stderr)

Return code: 0
Rank 3: deep interior and boundary ring PASS
Rank 1: deep interior and boundary ring PASS
Rank 0: deep interior and boundary ring PASS
Rank 2: deep interior and boundary ring PASS



In [62]:
import os
import subprocess

env = os.environ.copy()
env["OMPI_MCA_btl"] = "self,tcp"
env["OMPI_MCA_pml"] = "ob1"
env["OMPI_MCA_rmaps_base_oversubscribe"] = "1"

code = r'''
import numpy as np
from mpi4py import MPI
from numba import cuda
from distributed_cuda_solver import tiled_stencil_kernel

rank = MPI.COMM_WORLD.Get_rank()
cuda.select_device(0)

u = cuda.to_device(np.ones((258,258), dtype=np.float64))
k = cuda.to_device(np.full((258,258), 0.15))
s = cuda.to_device(np.zeros((258,258)))
out = cuda.device_array((258,258), dtype=np.float64)

# Compile the kernel first.
tiled_stencil_kernel[(16,16),(16,16)](
    u, k, s, out, 0.000001, 0
)
cuda.synchronize()

# Inspect generated PTX.
compiled = tiled_stencil_kernel.inspect_asm()
ptx = next(iter(compiled.values())) if isinstance(compiled, dict) else compiled

shared_loads = ptx.count("ld.shared")
shared_stores = ptx.count("st.shared")
barriers = ptx.count("bar.sync")

print(
    f"Rank {rank}: shared loads={shared_loads}, "
    f"shared stores={shared_stores}, barriers={barriers}",
    flush=True
)

assert shared_loads > 0, "No shared-memory loads found"
assert shared_stores > 0, "No shared-memory stores found"
assert barriers > 0, "No synchronization barrier found"

print(f"Rank {rank}: PTX AUDIT PASSED", flush=True)
'''

result = subprocess.run(
    [
        "mpiexec", "--allow-run-as-root", "--oversubscribe",
        "-n", "4", "/usr/bin/python3", "-c", code
    ],
    env=env,
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print("OUTPUT:")
print(result.stdout)

if result.stderr:
    print("ERRORS:")
    print(result.stderr)

Return code: 0
OUTPUT:
Rank 0: shared loads=5, shared stores=2, barriers=1
Rank 0: PTX AUDIT PASSED
Rank 2: shared loads=5, shared stores=2, barriers=1
Rank 2: PTX AUDIT PASSED
Rank 1: shared loads=5, shared stores=2, barriers=1
Rank 1: PTX AUDIT PASSED
Rank 3: shared loads=5, shared stores=2, barriers=1
Rank 3: PTX AUDIT PASSED



In [63]:
import os
import subprocess

env = os.environ.copy()
env["OMPI_MCA_btl"] = "self,tcp"
env["OMPI_MCA_pml"] = "ob1"
env["OMPI_MCA_rmaps_base_oversubscribe"] = "1"

result = subprocess.run(
    [
        "mpiexec",
        "--allow-run-as-root",
        "--oversubscribe",
        "-n", "4",
        "/usr/bin/python3",
        "distributed_cuda_solver.py",
        "belkamenbirgee"
    ],
    env=env,
    capture_output=True,
    text=True,
    timeout=1800
)

print("Return code:", result.returncode)
print("\nOUTPUT:")
print(result.stdout)

if result.stderr:
    print("\nERRORS:")
    print(result.stderr)

Return code: 0

OUTPUT:
Rank 0: coordinates=[0, 0], neighbors=(-2, 2, -2, 1)
Rank 2: coordinates=[1, 0], neighbors=(0, -2, -2, 3)
Rank 1: coordinates=[0, 1], neighbors=(-2, 3, 0, -2)
Rank 3: coordinates=[1, 1], neighbors=(1, -2, 2, -2)
\nStarting distributed CUDA stencil solver
MPI ranks: 4
Global grid: 512 x 512
Local grid: 256 x 256
Steps: 400
Maximum diffusivity: 0.23999962497115201
dt: 7.91634496767161e-07
serialized | Step 100 | Energy=1.181223808481e-01 | L2 residual=3.089903915164e-02
serialized | Step 200 | Energy=1.181133571454e-01 | L2 residual=1.862863208551e-02
serialized | Step 300 | Energy=1.181063029369e-01 | L2 residual=1.405244933967e-02
serialized | Step 400 | Energy=1.181002474116e-01 | L2 residual=1.160132277392e-02
overlapped | Step 100 | Energy=1.181223808481e-01 | L2 residual=3.089903915164e-02
overlapped | Step 200 | Energy=1.181133571454e-01 | L2 residual=1.862863208551e-02
overlapped | Step 300 | Energy=1.181063029369e-01 | L2 residual=1.405244933967e-02
overl

In [64]:
from pathlib import Path
import json

files = [
    "distributed_cuda_solver.py",
    "submission_manifest.json"
]

for filename in files:
    path = Path(filename)
    print(
        filename,
        "EXISTS" if path.exists() else "MISSING",
        f"({path.stat().st_size:,} bytes)" if path.exists() else ""
    )

with open("submission_manifest.json") as f:
    manifest = json.load(f)

print("\nFinal verification:")
print("Numerical match:", manifest["numerical_match_1e_12"])
print("Maximum difference:", manifest["max_final_state_difference"])
print("Serialized time:", manifest["serialized_time_seconds"])
print("Overlapped time:", manifest["overlapped_time_seconds"])
print("PTX audit:", manifest["ptx_audit"])

distributed_cuda_solver.py EXISTS (23,795 bytes)
submission_manifest.json EXISTS (1,067 bytes)

Final verification:
Numerical match: True
Maximum difference: 0.0
Serialized time: 8.63466966
Overlapped time: 4.470759577999999
PTX audit: {'shared_loads': 5, 'shared_stores': 2, 'bar_sync': 1, 'has_shared_memory': True, 'has_barrier': True}


In [66]:
from google.colab import files

files.download("distributed_cuda_solver.py")
files.download("submission_manifest.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>